# 9.4 · Keras 入门 / Keras Basics

> **课程定位 / Where this fits**
> 第 4 课，**Part 9 · 深度学习基础**。
> Lesson 4, **Part 9 · Deep Learning Foundations**.
>
> 9.3 的 PyTorch 要你**手写训练循环**——灵活但啰嗦。**Keras** 是另一大主流框架，走"高层 API"路线：`model.compile()` + `model.fit()` 几行就训完，把训练循环、指标、回调全替你封装好。它适合快速搭原型。现代 **Keras 3** 还能跑在 PyTorch/TensorFlow/JAX 任意后端上——本课就用 torch 后端。
> PyTorch (9.3) makes you **write the training loop by hand** — flexible but verbose. **Keras**, the other major framework, takes a "high-level API" route: `model.compile()` + `model.fit()` train in a few lines, wrapping the loop, metrics, and callbacks for you. Great for rapid prototyping. Modern **Keras 3** even runs on PyTorch/TensorFlow/JAX backends — here on the torch backend.
>
> 💼 **实战/面试视角**："Keras vs PyTorch 区别 / 高层 vs 低层 API / 什么时候用哪个" 是框架选型常考。
> 💼 **Practical/interview angle:** "Keras vs PyTorch / high- vs low-level API / when to use which" — framework-choice questions.

> 📐 **符号约定 / Notation**
> - Sequential API —— 线性堆叠层 / linear stack of layers
> - Functional API —— 用函数式连接层(支持多输入/分支)/ functional graph of layers

> 💡 **面试相关 / Interview-relevant**
> - "Keras 和 PyTorch 的区别 / 设计哲学"（出镜率 ★★★★）
> - "Sequential vs Functional API"（★★★★）
> - "compile/fit/evaluate 各做什么"（★★★）
> - "回调(callback)是什么 / 早停怎么写"（★★★）

---

## 学习目标 / Learning Objectives

1. 用 **Sequential API** 几行搭一个网络。
   Build a net in a few lines with the Sequential API.
2. 用 **compile/fit/evaluate** 完成训练（对比 PyTorch 的手写循环）。
   Train via compile/fit/evaluate (vs PyTorch's manual loop).
3. 用 **Functional API** 搭非线性结构（多输入/分支）。
   Build non-linear architectures (multi-input/branches) with the Functional API.
4. 用**回调(callbacks)** 做早停等。
   Use callbacks for early stopping, etc.
5. 理解 Keras vs PyTorch 的取舍。
   Understand the Keras vs PyTorch trade-off.

## 目录 / TOC
1. [先建直觉：高层 vs 低层 ⭐](#1)
2. [🔢 Sequential API + compile/fit ⭐](#2)
3. [Functional API：分支结构 ⭐](#3)
4. [回调：早停 ⭐](#4)
5. [Keras vs PyTorch + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：高层 vs 低层 ⭐ / Intuition: High- vs Low-level

同一个网络，PyTorch 和 Keras 的"训练"代码量差很多：
The same net needs very different amounts of "training" code in PyTorch vs Keras:
- **PyTorch（低层）**：你自己写训练循环（zero_grad→forward→loss→backward→step），完全掌控每一步——灵活，适合研究/自定义。
  **PyTorch (low-level):** you write the loop yourself (zero_grad→forward→loss→backward→step), full control of every step — flexible, great for research/custom work.
- **Keras（高层）**：`model.compile(优化器, 损失, 指标)` 配置好，再 `model.fit(X, y, epochs=...)` 一行训完，训练循环被封装在内——简洁，适合快速原型和标准任务。
  **Keras (high-level):** `model.compile(optimizer, loss, metrics)` configures it, then `model.fit(X, y, epochs=...)` trains in one line, the loop hidden inside — concise, great for prototyping and standard tasks.

两者没有优劣，是**控制力 vs 简洁性**的权衡。会两者都是加分项。本课用 **Keras 3 + torch 后端**（所以底层还是 PyTorch 在算，只是 API 是 Keras 风格）。
Neither is "better"; it's a **control vs conciseness** trade-off. Knowing both is a plus. We use **Keras 3 on the torch backend** (so PyTorch does the math underneath, with a Keras-style API on top).


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"        # 关键: 让 Keras 3 用 PyTorch 后端 / use torch backend
import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
keras.utils.set_random_seed(0)
print(f"Keras {keras.__version__}, 后端 backend = {keras.backend.backend()}")
print("Keras 3 可跑在 torch/tensorflow/jax 任意后端; 这里底层是 PyTorch 在算")


<a id="2"></a>
## 2. Sequential API + compile/fit ⭐ / Sequential API & compile/fit

**Sequential API** 把层**线性堆叠**起来，最直观。三步走：
The **Sequential API** stacks layers **linearly** — the most intuitive style. Three steps:
1. **建模**：`keras.Sequential([层1, 层2, ...])`。
   **Build:** `keras.Sequential([layer1, layer2, ...])`.
2. **`compile`**：指定优化器、损失、要监控的指标。
   **`compile`:** specify optimizer, loss, and metrics to track.
3. **`fit`**：传数据和 epochs，它自动完成整个训练循环（对比 9.3 你手写的五件套）。
   **`fit`:** pass data and epochs; it runs the whole training loop (vs the 5 manual steps in 9.3).


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data.astype("float32") / 16.0
y = digits.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

# 1) 建模: Sequential 线性堆叠层 / build by stacking layers
model = keras.Sequential([
    keras.layers.Input(shape=(64,)),
    keras.layers.Dense(64, activation="relu"),    # 全连接层(=nn.Linear+激活)
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(10, activation="softmax"), # 输出层直接带 softmax
])
# 2) compile: 配置优化器/损失/指标 / configure
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

# 3) fit: 一行完成整个训练循环(对比 9.3 手写五件套) / one line trains it all
hist = model.fit(X_tr, y_tr, epochs=30, batch_size=64, validation_split=0.2, verbose=0)
test_loss, test_acc = model.evaluate(X_te, y_te, verbose=0)
print(f"\nKeras Sequential on Digits: test 准确率 = {test_acc:.3f}")
print("compile + fit 两行就训完 — 训练循环被 Keras 封装(对比 9.3 手写五件套)")


In [ ]:
# fit 返回的 history 自动记录了训练过程 / history records the training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(hist.history["loss"], label="train"); axes[0].plot(hist.history["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("损失曲线")
axes[1].plot(hist.history["accuracy"], label="train"); axes[1].plot(hist.history["val_accuracy"], label="val")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(); axes[1].set_title("准确率曲线")
plt.tight_layout(); plt.show()
print("fit 自动记录 train/val 的 loss 和 metrics → 直接画学习曲线(对比 PyTorch 要自己存)")


<a id="3"></a>
## 3. Functional API：分支结构 ⭐ / Functional API for Branches

Sequential 只能"一条线"堆叠。当网络有**多输入、多输出、或分支/跳连**（如残差连接、双塔模型）时，要用 **Functional API**：把每层当成一个**函数**作用在张量上，显式连接，能搭出任意有向无环图。
Sequential only stacks "in a line". For **multiple inputs/outputs or branches/skip connections** (residual connections, two-tower models), use the **Functional API**: treat each layer as a **function** applied to tensors, connecting them explicitly to build any directed acyclic graph.

下面用 Functional API 搭一个有**两个分支**再合并的网络作演示。
Below we build a network with **two branches** that merge, demonstrating the Functional API.


In [ ]:
# Functional API: 把层当函数作用在张量上, 显式连接 / layers as functions on tensors
inputs = keras.Input(shape=(64,))
branch_a = keras.layers.Dense(32, activation="relu")(inputs)    # 分支 A
branch_b = keras.layers.Dense(32, activation="tanh")(inputs)    # 分支 B(不同激活)
merged = keras.layers.Concatenate()([branch_a, branch_b])       # 合并两分支
x = keras.layers.Dense(32, activation="relu")(merged)
outputs = keras.layers.Dense(10, activation="softmax")(x)
func_model = keras.Model(inputs, outputs)                       # 指定输入输出构成模型

func_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
func_model.fit(X_tr, y_tr, epochs=30, batch_size=64, verbose=0)
print(f"Functional API(双分支) on Digits: test 准确率 = {func_model.evaluate(X_te, y_te, verbose=0)[1]:.3f}")
print("Functional API 支持多输入/多输出/分支/跳连(残差等) → Sequential 做不到的复杂结构")


<a id="4"></a>
## 4. 回调：早停 ⭐ / Callbacks: Early Stopping

**回调(callbacks)** 是 Keras 的一大便利：在训练过程的特定时刻（每个 epoch 后等）自动执行某些操作，无需改训练代码。最常用的是**早停(EarlyStopping)**：监控验证损失，连续若干 epoch 不改善就**自动停止**并恢复最佳权重——省时间又防过拟合（接 7.3）。其它常用回调：`ModelCheckpoint`（存最佳模型）、`ReduceLROnPlateau`（自动降学习率，9.8）。
**Callbacks** are a Keras convenience: run actions at specific moments (after each epoch, etc.) without changing training code. The most common is **EarlyStopping**: monitor validation loss and **auto-stop** when it stops improving for several epochs, restoring the best weights — saving time and preventing overfitting (7.3). Others: `ModelCheckpoint` (save the best model), `ReduceLROnPlateau` (auto-lower LR, 9.8).


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)   # 验证损失5轮不降就停, 恢复最佳权重

model2 = keras.Sequential([
    keras.layers.Input(shape=(64,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# 设大上限 epochs=200, 让早停自己决定何时停 / large cap, let early stopping decide
hist2 = model2.fit(X_tr, y_tr, epochs=200, batch_size=64, validation_split=0.2,
                   callbacks=[early_stop], verbose=0)
print(f"早停: 上限设 200 epochs, 实际只训了 {len(hist2.history['loss'])} 轮就停了")
print(f"test 准确率 = {model2.evaluate(X_te, y_te, verbose=0)[1]:.3f}")
print("EarlyStopping 自动在验证损失不再改善时停止+恢复最佳权重 → 省时+防过拟合(7.3)")


<a id="5"></a>
## 5. Keras vs PyTorch + 小结 ⭐ / Keras vs PyTorch & Summary

| | PyTorch (9.3) | Keras |
|---|---|---|
| 风格 style | 低层, **手写训练循环** | 高层, **compile + fit** |
| 控制力 control | 完全掌控每一步 | 训练循环被封装 |
| 代码量 verbosity | 多 | 少 |
| 适合 best for | 研究/自定义/调试 | 快速原型/标准任务 |
| 学术界主流 | **PyTorch 占主导** | Keras 在工程/教学常见 |

**实战建议**：研究、需要自定义训练逻辑、或要看每一步细节 → PyTorch；快速搭标准模型、原型验证 → Keras。两者会用都是加分项。注意 PyTorch 现在也有 **Lightning** 等高层封装、Keras 3 也能跑 torch 后端——两种范式在融合。
**Practical advice:** research, custom training logic, or step-level debugging → PyTorch; quick standard models and prototyping → Keras. Knowing both is a plus. Note PyTorch now has high-level wrappers (Lightning) and Keras 3 runs on torch — the two paradigms are converging.

```
Keras: 高层 API; Sequential(线性堆叠) / Functional(分支/多输入跳连); compile→fit→evaluate
fit 一行封装整个训练循环 + 自动记录 history(对比 9.3 PyTorch 手写五件套)
callbacks: 训练中自动执行(EarlyStopping 早停/ModelCheckpoint 存最佳/ReduceLROnPlateau)
Keras vs PyTorch: 简洁(fit) vs 控制力(手写循环); 原型用 Keras, 研究/自定义用 PyTorch
Keras 3 可跑 torch/tf/jax 后端; 两种范式在融合(PyTorch Lightning 等)
```

### 💡 面试速查 / Interview cheat-sheet
1. **Keras 高层(compile+fit) vs PyTorch 低层(手写循环)**: 简洁 vs 控制力。
   Keras high-level (compile+fit) vs PyTorch low-level (manual loop): conciseness vs control.
2. **Sequential(线性堆叠) vs Functional(分支/多输入/跳连)**。
   Sequential (linear stack) vs Functional (branches/multi-input/skip).
3. **compile 配置(优化器/损失/指标), fit 训练, evaluate 评估**。
   compile configures (optimizer/loss/metrics), fit trains, evaluate tests.
4. **callbacks 自动执行**(早停/存最佳/降 LR), 无需改训练代码。
   Callbacks run automatically (early stop/checkpoint/LR decay) without touching training code.
5. **研究用 PyTorch, 原型用 Keras**; Keras 3 可跑 torch 后端。
   PyTorch for research, Keras for prototyping; Keras 3 runs on the torch backend.

### 下一节 / Next
**9.5 激活函数**——网络的非线性来源。sigmoid/tanh/ReLU/LeakyReLU/GELU/Swish 的形状、梯度、优缺点, 以及"为什么 ReLU 取代了 sigmoid"。
**9.5 Activations** — the source of a network's nonlinearity. Shapes, gradients, and trade-offs of sigmoid/tanh/ReLU/LeakyReLU/GELU/Swish, and "why ReLU replaced sigmoid".
